In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
os.environ["SHAPE_RESTORE_SHX"] = "YES"

import election_sim_partisan as esp
import election_sim_racial as esr

In [ ]:
DATA_DIR = os.environ.get("FRA_DATA_DIR", "./data")

state = "MA"
prec_shp = f"{DATA_DIR}/MA/ma_2024_gen_prec/ma_2024_gen_all_prec/ma_2024_gen_all_prec_updated.shp"
prec_shp_race = f"{DATA_DIR}/MA/ma_2024_gen_prec/ma_2024_gen_all_prec/ma_2024_gen_all_prec.shp"  # racial sim uses a different precinct file than partisan (confirmed across all states)
block_shp = f"{DATA_DIR}/MA/tl_2020_25_tabblock20/tl_2020_25_tabblock20.shp"
comb_summary = f"{DATA_DIR}/MA/MA_combinenew.csv"

party_list = ["dem", "rep"]
race_list  = ["white", "asian", "latino", "black", "native"]
pi = {'white': 0.6, 'asian': 0.9, 'latino': 0.9, 'black': 0.9, 'native': 0.9}  # in-group loyalty

# NOTE: this state's racial notebook used a different district/seat structure than its
# partisan notebook (num_elected matched, num_districts/num_seats/num_cands did not) --
# preserved separately rather than forced to match.
num_districts_race = 8
num_seats_race   = 5
num_cands_race   = 10
cand_list_race   = [f"C{i}" for i in range(1, num_cands_race * num_districts_race + 1)]
num_districts = 10
num_seats   = 4          # seats per district
num_cands   = 8         # candidates per district (2x seats gives real RCV competition)
num_elected = 40
cand_list   = [f"C{i}" for i in range(1, num_cands * num_districts + 1)]
pop_col = 'TOTPOP'
total_steps = 100
crs = "EPSG:32619"
crs_race = "EPSG:32619"
num_district_c = 40
num_cands_c = 2
num_seats_c = 1


In [ ]:
# ══════════════════════════ Partisan simulation ══════════════════════════
gdfprec_merged = esp.data_merge(block_shp, prec_shp, state, crs)
# gdfprec_merged is a precinct-level GeoDataFrame with election + demographic columns, ready for Graph.from_geodataframe / build_chain.
sampled_maps = esp.build_chain(pop_col, gdfprec_merged, num_districts)
wn, df_party_cands, df_vote_portion, election_margins_df = esp.election_sim(
    sampled_maps, num_districts, num_seats, num_cands, cand_list, party_list
)
seat_shares = esp.compute_seat_shares(wn, df_party_cands, num_districts, num_elected, party_list)
sampled_maps_c = esp.build_chain(pop_col, gdfprec_merged, num_district_c)
wn_c, df_party_cands_c, df_vote_portion_c, election_margins_df_c = esp.election_sim_c(
    sampled_maps_c, num_district_c, num_cands_c, cand_list, party_list
)

In [ ]:
state_props = esp.compute_state_props(gdfprec_merged, party_list)
status_quo = {'dem': 0.875, 'rep': 0.125}  # MA status quo senate seat share
state_info_p = {"state_props": state_props, "status_quo": status_quo}

esp.plot_seat_shares(seat_shares, state_props, status_quo, party_list)

In [ ]:
esp.plot_competitiveness_violin(election_margins_df, election_margins_df_c, state)

In [ ]:
esp.plot_competitiveness_cdf(election_margins_df, election_margins_df_c, state)

In [ ]:
base = f"{DATA_DIR}/{state}"
os.makedirs(base, exist_ok=True)

with open(f"{base}/{state}_sl_wn_p.pkl", "wb") as f:
    pickle.dump(wn, f)
with open(f"{base}/{state}_sl_seat_shares_p.pkl", "wb") as f:
    pickle.dump(seat_shares, f)

maps_assignments = [dict(p.assignment) for p in sampled_maps]
with open(f"{base}/{state}_sl_sampled_maps_assignments_p.pkl", "wb") as f:
    pickle.dump(maps_assignments, f)

with open(f"{base}/{state}_state_info_p.pkl", "wb") as f:
    pickle.dump(state_info_p, f)

with open(f"{base}/{state}_election_margins_rcv.pkl", "wb") as f:
    pickle.dump(election_margins_df, f)

with open(f"{base}/{state}_election_margins_smd.pkl", "wb") as f:
    pickle.dump(election_margins_df_c, f)

print("Saved partisan results to", base)

In [ ]:
seat_shares_smd = esp.compute_seat_shares_swc(wn_c, df_party_cands_c, num_district_c, num_elected, party_list)

with open(f"{base}/{state}_sl_seat_shares_smd.pkl", "wb") as f:
    pickle.dump(seat_shares_smd, f)

print("Saved partisan SMD seat shares to", base)

In [ ]:
# ══════════════════════════ Racial simulation ══════════════════════════
gdfprec_merged_race = esr.data_merge(block_shp, prec_shp_race, comb_summary, state, crs_race)
sampled_maps_race = esr.build_chain(pop_col, gdfprec_merged_race, num_districts_race)
wn_race, df_race_cands, df_vote_portion_race, election_margins_df_race = esr.election_sim(
    sampled_maps_race, num_districts_race, num_seats_race, num_cands_race, cand_list_race, race_list, pi
)
seat_shares_race = esr.compute_seat_shares(wn_race, df_race_cands, num_districts_race, num_elected, race_list)
sampled_maps_c_race = esr.build_chain(pop_col, gdfprec_merged_race, num_district_c)
wn_c_race, df_race_cands_c, df_vote_portion_c_race, election_margins_df_c_race = esr.election_sim_c(
    sampled_maps_c_race, num_district_c, num_cands_c, cand_list_race, race_list, pi
)

In [ ]:
state_props_race = esr.compute_state_props(gdfprec_merged_race, race_list)
status_quo_race = {'white': 0.87, 'asian': 0.03, 'latino': 0.07, 'black': 0.02, 'native': 0.01}  # MA status quo senate seat share
state_info_race = {"state_props": state_props_race, "status_quo": status_quo_race}

esr.plot_seat_shares(seat_shares_race, state_props_race, status_quo_race, race_list)

In [ ]:
esr.plot_competitiveness_violin(election_margins_df_race, election_margins_df_c_race, state)

In [ ]:
esr.plot_competitiveness_cdf(election_margins_df_race, election_margins_df_c_race, state)

In [ ]:
with open(f"{base}/{state}_sl_wn_race.pkl", "wb") as f:
    pickle.dump(wn_race, f)
with open(f"{base}/{state}_sl_seat_shares_race.pkl", "wb") as f:
    pickle.dump(seat_shares_race, f)

maps_assignments_race = [dict(p.assignment) for p in sampled_maps_race]
with open(f"{base}/{state}_sl_sampled_maps_assignments_race.pkl", "wb") as f:
    pickle.dump(maps_assignments_race, f)

with open(f"{base}/{state}_state_info_race.pkl", "wb") as f:
    pickle.dump(state_info_race, f)

with open(f"{base}/{state}_election_margins_rcv_race.pkl", "wb") as f:
    pickle.dump(election_margins_df_race, f)

with open(f"{base}/{state}_election_margins_smd_race.pkl", "wb") as f:
    pickle.dump(election_margins_df_c_race, f)

print("Saved racial results to", base)

In [ ]:
seat_shares_smd_race = esr.compute_seat_shares_swc(wn_c_race, df_race_cands_c, num_district_c, num_elected, race_list)

with open(f"{base}/{state}_sl_seat_shares_smd_race.pkl", "wb") as f:
    pickle.dump(seat_shares_smd_race, f)

print("Saved racial SMD seat shares to", base)